# Lab 2A: Tokens, embeddings and semantic search
**Presenter: Nagarjuna Naidu**

Story of this lab: **text -> tokens -> token ids -> vectors -> similarity -> search -> vector database**

**Learning outcomes**
- See how text becomes tokens and ids, and why token counts matter (cost and limits).
- Turn text into 384-d vectors, measure similarity, and visualise "meaning as geometry".
- Build semantic search over 48 documents, compare it with keyword search, and store it in a vector database (Chroma) with metadata filters.

**Time budget:** ~40 min core (sections 1-6). Weaviate (section 7) and the negation puzzle (section 8) are OPTIONAL.

In [ ]:
import sys, platform, random, warnings, os, json, re, pathlib, shutil
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import sklearn, chromadb, fastembed, tokenizers

random.seed(0); np.random.seed(0)
INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"
plt.rcParams.update({
    "axes.prop_cycle": matplotlib.cycler(color=[TEAL, CORAL, GOLD, PURPLE, LIME, INK]),
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
})
pathlib.Path("artifacts").mkdir(exist_ok=True)
print("python", platform.python_version(), "| numpy", np.__version__, "| sklearn", sklearn.__version__,
      "| chromadb", chromadb.__version__, "| fastembed", fastembed.__version__, "| tokenizers", tokenizers.__version__)
print("cwd:", os.getcwd())

## 1. Setup: load the embedding model and the corpus once
- Model: `BAAI/bge-small-en-v1.5` (384-d, runs locally on CPU, no API key).
- Corpus: 48 company documents, each with a header (`title`, `department`, `date`, `sensitivity`...) and a body.

In [ ]:
import time
from fastembed import TextEmbedding

t0 = time.time()
model = TextEmbedding("BAAI/bge-small-en-v1.5")
embed = lambda texts: np.array(list(model.embed(list(texts))))   # list[str] -> (n, 384) array
embed(["warm-up"])
print(f"model ready in {time.time()-t0:.1f}s | vector size = {model.embedding_size}")

def load_doc(path):
    head, body = path.read_text(encoding="utf-8").split("\n\n", 1)
    meta = dict(l.split(": ", 1) for l in head.splitlines() if ": " in l)
    meta.update(doc_id=path.name.split("_")[0], body=body.strip())
    return meta

DOCS = [load_doc(p) for p in sorted(pathlib.Path("data/corpus").glob("*.txt"))]
print(len(DOCS), "documents | departments:", sorted({d["department"] for d in DOCS}))
print("example:", DOCS[4]["doc_id"], "-", DOCS[4]["title"])

## 2. Tokens: what the model actually sees
- Slide recap: text is split into **tokens**, each token gets an **id**; the model only ever sees ids.
- We use the embedding model's *own* tokenizer (WordPiece). `##` marks "continues the previous piece".

In [ ]:
from tokenizers import Tokenizer
# fastembed's own WordPiece tokenizer; we take a copy so our settings do not affect the embedding model
tok = Tokenizer.from_str(model.model.tokenizer.to_str())
tok.no_padding(); tok.no_truncation()

def tokenize(text, special=False):
    enc = tok.encode(text, add_special_tokens=special)
    return enc.tokens, enc.ids

sentence = "I love natural language processing"
toks, ids = tokenize(sentence, special=True)
display(pd.DataFrame({"token": toks, "id": ids}).T)
print("[CLS] / [SEP] are special markers the model adds at the start / end of the text.")

In [ ]:
# Word vs character vs subword
words = ["unhappiness", "tokenization", "pneumonoultramicroscopic"]
rows = []
for w in words:
    sub, _ = tokenize(w)
    rows.append({"word": w, "word-level": 1, "character-level": len(w), "subword (WordPiece)": len(sub), "pieces": " + ".join(sub)})
display(pd.DataFrame(rows))
print("Word-level: huge vocabulary, unknown words break it. Character-level: tiny vocabulary, very long sequences.")
print("Subword: a middle path. Common words stay whole, rare words are assembled from pieces.")

In [ ]:
# Tokens vs words = cost and limits
para = DOCS[4]["body"].split("\n\n")[1] if len(DOCS[4]["body"].split("\n\n")) > 1 else DOCS[4]["body"][:600]
n_words, n_tokens = len(para.split()), len(tokenize(para)[0])
print(para[:300] + " ...\n")
print(f"words = {n_words} | tokens = {n_tokens} | tokens per word = {n_tokens/n_words:.2f}")

all_w = [len(d["body"].split()) for d in DOCS]; all_t = [len(tokenize(d["body"])[0]) for d in DOCS]
print(f"Whole corpus: {sum(all_w):,} words -> {sum(all_t):,} tokens (x{sum(all_t)/sum(all_w):.2f}). "
      f"Longest doc = {max(all_t)} tokens vs this model's 512-token input limit.")
print("LLM APIs bill and limit by tokens, not words. Tokenizers differ per model, so counts differ too.")

## 3. Embeddings: text -> vector
- An embedding model maps each text to a point in 384-d space. Similar meaning = nearby points.
- **Cosine similarity** compares directions: 1 = same, 0 = unrelated.

In [ ]:
items = ["cat", "kitten", "dog", "car", "truck", "bank loan", "mortgage", "river bank", "pizza", "I love natural language processing"]
V = embed(items)
print("shape:", V.shape, "-> one 384-number vector per item")
print("first 6 numbers of 'cat':", np.round(V[0, :6], 3), "| length of vector:", round(float(np.linalg.norm(V[0])), 3))

def cosine(a, b):
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    return (a / np.linalg.norm(a, axis=1, keepdims=True)) @ (b / np.linalg.norm(b, axis=1, keepdims=True)).T

print("cos(cat, kitten) =", round(float(cosine(V[0], V[1])[0, 0]), 3), "| cos(cat, truck) =", round(float(cosine(V[0], V[4])[0, 0]), 3))

In [ ]:
from sklearn.decomposition import PCA
S = cosine(V, V)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.2))
im = a1.imshow(S, cmap="viridis", vmin=0.4, vmax=1)
a1.set_xticks(range(len(items))); a1.set_yticks(range(len(items)))
a1.set_xticklabels([i[:14] for i in items], rotation=60, ha="right"); a1.set_yticklabels([i[:14] for i in items])
for i in range(len(items)):
    for j in range(len(items)):
        a1.text(j, i, f"{S[i,j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if S[i, j] < 0.75 else INK)
a1.set_title("Cosine similarity heatmap"); plt.colorbar(im, ax=a1, fraction=0.046)
P = PCA(2, random_state=0).fit_transform(V)
a2.scatter(P[:, 0], P[:, 1], s=70, color=TEAL)
for (x, y), t in zip(P, items): a2.annotate(t[:22], (x, y), xytext=(5, 4), textcoords="offset points", fontsize=9)
a2.set_title("2-D PCA projection (384 dims squashed to 2)"); a2.set_xlabel("PC1"); a2.set_ylabel("PC2")
plt.tight_layout(); plt.show()

### Your turn 1: "Where would you place Apple?"
- Same exercise as the slide. Embed these 10 items, plot them in 2-D, and look at where `apple` lands.
- Hint: reuse `embed`, `PCA` and the plotting pattern above. Colour fruit teal, places coral, tech gold, `apple` purple.

In [ ]:
apple_items = ["cherry", "watermelon", "apartment", "school", "hospital", "apple", "iphone", "microsoft", "apple pie", "apple stock price"]
# TODO: V2 = embed(...)  ->  P2 = PCA(2).fit_transform(V2)  ->  scatter + annotate each point
# Hint: colours = {"fruit": TEAL, "place": CORAL, "tech": GOLD}
print("Items to plot:", apple_items)

**Discussion**
- Single-word `apple` sits *between* fruit and technology: one word, several meanings.
- Whole sentences carry context, so their vectors land clearly on one side. This is why we embed passages, not isolated words.

## 4. Export for the TensorFlow Embedding Projector
- Embed each of the 48 documents (`title` + first ~120 words), write two TSV files, then explore them in 3-D in your browser.

In [ ]:
def doc_text(d, n=120):
    return d["title"] + ". " + " ".join(d["body"].split()[:n])

t0 = time.time()
DOC_TEXT = [doc_text(d) for d in DOCS]
DOC_VECS = embed(DOC_TEXT)
print("doc matrix:", DOC_VECS.shape, f"({time.time()-t0:.1f}s)")

pdir = pathlib.Path("artifacts/projector"); pdir.mkdir(parents=True, exist_ok=True)
np.savetxt(pdir / "vectors.tsv", DOC_VECS, delimiter="\t", fmt="%.6f")
meta = pd.DataFrame({"title": [d["title"] for d in DOCS], "department": [d["department"] for d in DOCS], "sensitivity": [d["sensitivity"] for d in DOCS]})
meta.to_csv(pdir / "metadata.tsv", sep="\t", index=False)
print("wrote", pdir / "vectors.tsv", "and", pdir / "metadata.tsv")
print("""
Steps (live at https://projector.tensorflow.org):
  1. Click 'Load' (left panel).   2. Step 1: choose artifacts/projector/vectors.tsv.
  3. Step 2: choose artifacts/projector/metadata.tsv.   4. Click outside the dialog.
  5. Bottom-left: try PCA, then t-SNE, then UMAP.   6. 'Color by' -> department (or sensitivity).
  Question: do documents cluster by department, or by topic? Which HR and Legal docs sit together?""")

In [ ]:
# In-notebook fallback: the same picture as a static 2-D scatter
Pd = PCA(2, random_state=0).fit_transform(DOC_VECS)
dept_cols = dict(zip(sorted(meta.department.unique()), [TEAL, CORAL, GOLD, PURPLE, LIME, INK, "#7A8B94", "#C0392B"]))
fig, ax = plt.subplots(figsize=(10, 6))
for dep, c in dept_cols.items():
    m = (meta.department == dep).values
    ax.scatter(Pd[m, 0], Pd[m, 1], s=60, color=c, label=dep, edgecolor="white")
for i, d in enumerate(DOCS): ax.annotate(d["doc_id"], (Pd[i, 0], Pd[i, 1]), xytext=(3, 3), textcoords="offset points", fontsize=7)
ax.legend(title="department", frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("48 corpus documents, PCA of their embeddings"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

## 5. Semantic search vs keyword search
- **Semantic**: embed the query, rank documents by cosine similarity.
- **Keyword baseline**: TF-IDF (counts shared words).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
FULL_TEXT = [d["title"] + ". " + d["body"] for d in DOCS]
tfidf = TfidfVectorizer(stop_words="english", token_pattern=r"(?u)\b\w[\w-]*\w\b|\b\w\b", sublinear_tf=True)
TFIDF_M = tfidf.fit_transform(FULL_TEXT)
IDS = [d["doc_id"] for d in DOCS]

def semantic_search(query, k=3):
    s = cosine(embed([query]), DOC_VECS)[0]
    return [(IDS[i], float(s[i])) for i in np.argsort(-s)[:k]]

def keyword_search(query, k=3):
    s = (TFIDF_M @ tfidf.transform([query]).T).toarray().ravel()
    return [(IDS[i], float(s[i])) for i in np.argsort(-s)[:k] if s[i] > 0]

TITLE = {d["doc_id"]: d["title"] for d in DOCS}
showcase = ["can I work from Spain for a month", "my laptop broke, what now", "can I paste client data into ChatGPT", "INC-22007"]
rows = []
for q in showcase:
    for name, fn in (("semantic", semantic_search), ("keyword", keyword_search)):
        res = fn(q)
        rows.append({"query": q, "method": name, "top-3": " | ".join(f"{i} {TITLE[i][:34]}" for i, _ in res) or "(no match)"})
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 250)
display(pd.DataFrame(rows).set_index(["query", "method"]))
print("Check: D05 (remote work abroad) never contains 'Spain' or 'month'. The ticket id is an exact string: keyword wins there.")

In [ ]:
# Tiny evaluation: Recall@3 (doc level) on the gold queries
gold = [g for g in json.load(open("data/eval_queries.json")) if g["type"] in ("semantic", "keyword")]
def recall_at_k(fn, qs, k=3):
    return float(np.mean([len({i for i, _ in fn(g["query"], k)} & set(g["relevant_doc_ids"])) / len(g["relevant_doc_ids"]) for g in qs]))
out = []
for typ in ("semantic", "keyword"):
    qs = [g for g in gold if g["type"] == typ]
    out.append({"query type": typ, "n queries": len(qs), "semantic Recall@3": round(recall_at_k(semantic_search, qs), 2),
                "keyword Recall@3": round(recall_at_k(keyword_search, qs), 2)})
res_df = pd.DataFrame(out); display(res_df)
ax = res_df.set_index("query type")[["semantic Recall@3", "keyword Recall@3"]].plot.bar(color=[TEAL, CORAL], figsize=(6, 3.5), rot=0)
ax.set_ylim(0, 1.05); ax.set_title("Each method wins on its home turf"); ax.legend(frameon=False); plt.tight_layout(); plt.show()
print("Takeaway: neither is enough alone -> hybrid search (Lab 2C) combines them.")

## 6. Vector database with Chroma
- A vector DB stores vectors + metadata and finds nearest neighbours fast, and **persists** them to disk.
- **HNSW** in 3 bullets:
  - Builds a multi-layer graph; each vector links to its near neighbours.
  - A query hops greedily from sparse top layers down to dense bottom layers.
  - Approximate: far fewer comparisons than scanning all vectors, with a small recall trade-off.

In [ ]:
CHROMA_DIR = "artifacts/chroma"
shutil.rmtree(CHROMA_DIR, ignore_errors=True)                     # clean rebuild every run
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.create_collection("company_docs", metadata={"hnsw:space": "cosine"})
col.add(ids=IDS, embeddings=DOC_VECS.tolist(), documents=DOC_TEXT,
        metadatas=[{"title": d["title"], "department": d["department"], "sensitivity": d["sensitivity"], "date": d["date"]} for d in DOCS])
print("stored", col.count(), "docs in", CHROMA_DIR)

def chroma_search(query, k=3, where=None):
    r = col.query(query_embeddings=embed([query]).tolist(), n_results=k, where=where)   # SAME model as the index
    return pd.DataFrame({"id": r["ids"][0], "title": [m["title"][:48] for m in r["metadatas"][0]],
                         "dept": [m["department"] for m in r["metadatas"][0]], "sensitivity": [m["sensitivity"] for m in r["metadatas"][0]],
                         "distance": np.round(r["distances"][0], 3), "similarity": np.round(1 - np.array(r["distances"][0]), 3)})

q = "can I work from Spain for a month"
print("No filter:");                 display(chroma_search(q))
print("where department == Legal:"); display(chroma_search(q, where={"department": "Legal"}))
print("Chroma returns cosine DISTANCE; similarity = 1 - distance. The filter lifts the relevant Legal doc D05 to rank 1. (Similarity also ignores freshness: D07 is the outdated 2022 policy.)")

**Why this matters**
- Use the **same embedding model** for indexing and querying, or the vectors are not comparable.
- **Metadata filtering** (`where`) narrows by department, sensitivity, date *before* ranking: essential for access control and freshness.
- **Persistence**: the index survives restarts (`artifacts/chroma`), so you embed once, not on every run.

### Your turn 2: filter by metadata
- Return only documents with `sensitivity == "internal"` AND `department == "HR"` for the query "how many days can I work from home".
- Hint: combine conditions with `{"$and": [ {...}, {...} ]}`.

In [ ]:
q2 = "how many days can I work from home"
where_hr_internal = None   # TODO: replace with a Chroma `where` filter dict
display(chroma_search(q2, k=3, where=where_hr_internal))

## 7. OPTIONAL: the same index in Weaviate (embedded)
- Bring-your-own vectors, same 48 docs. Needs a one-time binary download and does not run on Windows; the cell skips gracefully.

In [ ]:
import contextlib
@contextlib.contextmanager
def quiet_fds():
    """Silence the embedded server's console logs (they are written by a child process)."""
    sys.stdout.flush(); sys.stderr.flush()
    saved = [os.dup(1), os.dup(2)]; dn = os.open(os.devnull, os.O_WRONLY)
    os.dup2(dn, 1); os.dup2(dn, 2)
    try: yield
    finally:
        os.dup2(saved[0], 1); os.dup2(saved[1], 2); [os.close(f) for f in saved + [dn]]

RUN_WEAVIATE = False   # set True to run this optional section (one-time server download; not supported on Windows)
result = None
try:
    if not RUN_WEAVIATE: raise RuntimeError("RUN_WEAVIATE is False; set it to True to run this optional section")
    import weaviate
    from weaviate.classes.config import Configure, Property, DataType
    from weaviate.classes.data import DataObject
    warnings.simplefilter("ignore")
    shutil.rmtree("artifacts/weaviate", ignore_errors=True)
    with quiet_fds(): wc = weaviate.connect_to_embedded(persistence_data_path="artifacts/weaviate", version="1.27.0")
    try:
        wc.collections.delete("CompanyDocs")
        w = wc.collections.create("CompanyDocs", vectorizer_config=Configure.Vectorizer.none(),
                                  properties=[Property(name="title", data_type=DataType.TEXT), Property(name="doc_id", data_type=DataType.TEXT)])
        w.data.insert_many([DataObject(properties={"title": d["title"], "doc_id": d["doc_id"]}, vector=v.tolist()) for d, v in zip(DOCS, DOC_VECS)])
        hits = w.query.near_vector(embed(["can I work from Spain for a month"])[0].tolist(), limit=3)
        result = [(o.properties["doc_id"], o.properties["title"][:40]) for o in hits.objects]
    finally:
        with quiet_fds(): wc.close()
    print("Weaviate top-3 (same vectors, same answer as Chroma):", result)
except Exception as e:
    print("Weaviate section skipped:", type(e).__name__, "-", str(e)[:120], "\n(This is fine: Chroma above covers the core ideas.)")

## 8. OPTIONAL stretch, Your turn 3: fool the semantic search
- Embeddings capture *topic* more than *logic*. Try negation: `can I claim alcohol` vs `cannot claim alcohol`.
- Task: compare the top-3 results and the query-to-query similarity. What does that tell you?

In [ ]:
neg_a, neg_b = "can I claim alcohol on expenses", "alcohol cannot be claimed on expenses"
# TODO: compute cosine(embed(neg_a), embed(neg_b)) and compare semantic_search(neg_a) with semantic_search(neg_b)
print(neg_a, "|", neg_b)

## 9. Recap and what's next
- Text is split into **subword tokens** and mapped to ids; token counts drive cost and limits.
- An **embedding** is a vector where nearby = similar meaning; **cosine similarity** measures it.
- **Semantic search** finds paraphrases keyword search misses; keyword search wins on exact IDs and codes.
- A **vector database** (Chroma, Weaviate) adds fast approximate search (HNSW), metadata filters and persistence.
- Embeddings capture topic, not logic (negation, dates, versions): retrieval is the start, not the whole answer.

**Next:** Lab 2E builds RAG: retrieve the right passages, then let an LLM answer from them with citations.